# فحص زوج صور الأقمار الصناعية قبل التقطيع

هذه الـNotebook تفحص ملفي:

- **MS_CLIP.tif**: الصورة متعددة الأطياف `MS`، وغالبًا هي الأقل دقة مكانية.
- **PAN_CLIP_TIFF.tif**: الصورة البانكروماتية `PAN`، وغالبًا هي الأعلى دقة مكانية.

> **مهم:** لا نبدأ التقطيع قبل التأكد من الأبعاد، عدد الـBands، الـCRS، الدقة المكانية، الحدود الجغرافية، ونسبة التداخل.

الـNotebook لا تعدّل الصور الأصلية ولا تحذف ملفات `.enp`.


## 1) تثبيت المكتبات

شغّل الخلية التالية مرة واحدة فقط. لو المكتبات موجودة بالفعل، لن يحدث ضرر.

In [ ]:
%pip install -q rasterio numpy pandas matplotlib scikit-image

## 2) كتابة مسارات الصور

عدّل المسارين فقط بما يطابق مكان الملفات عندك.

**ملاحظة:** اكتب حرف `r` قبل المسار، كما في المثال، حتى يتعامل Python مع مسارات Windows بصورة صحيحة.

In [ ]:
from pathlib import Path

# عدّل المسارين فقط
MS_PATH = Path(r"F:\بروجكت الهيئه\Super_Resolution_28-07-2026\Law_Resolution\MS_CLIP.tif")
PAN_PATH = Path(r"F:\بروجكت الهيئه\Super_Resolution_28-07-2026\High_Resolution\PAN_CLIP_TIFF.tif")

print("MS exists :", MS_PATH.exists(), "->", MS_PATH)
print("PAN exists:", PAN_PATH.exists(), "->", PAN_PATH)

if not MS_PATH.exists():
    print("\nعدّل MS_PATH لأن الملف غير موجود في المسار المكتوب.")
if not PAN_PATH.exists():
    print("\nعدّل PAN_PATH لأن الملف غير موجود في المسار المكتوب.")

## 3) قراءة معلومات كل صورة

سنستخرج:

- العرض والارتفاع بالبكسل.
- عدد الـBands.
- نوع البيانات.
- نظام الإحداثيات `CRS`.
- حجم البكسل.
- الحدود الجغرافية.
- قيمة NoData.
- بيانات التاريخ الموجودة داخل الملف، إن وُجدت.


In [ ]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.warp import transform_bounds

def image_info(path):
    with rasterio.open(path) as src:
        tags = src.tags()
        info = {
            "file": path.name,
            "width_px": src.width,
            "height_px": src.height,
            "bands": src.count,
            "dtype": ", ".join(sorted(set(src.dtypes))),
            "crs": str(src.crs),
            "pixel_size_x": abs(src.transform.a),
            "pixel_size_y": abs(src.transform.e),
            "left": src.bounds.left,
            "bottom": src.bounds.bottom,
            "right": src.bounds.right,
            "top": src.bounds.top,
            "nodata": src.nodata,
        }
    return info, tags

if not MS_PATH.exists() or not PAN_PATH.exists():
    raise FileNotFoundError("صحّح المسارات في الخلية السابقة ثم أعد التشغيل.")

ms_info, ms_tags = image_info(MS_PATH)
pan_info, pan_tags = image_info(PAN_PATH)

info_df = pd.DataFrame([ms_info, pan_info]).set_index("file")
display(info_df.T)


## 4) عرض الـMetadata والبحث عن تاريخ الالتقاط

وجود تاريخ داخل الـTIFF ليس مضمونًا. لو لم يظهر تاريخ واضح، سنحتاج ملف الـmetadata الأصلي من مزود الصورة أو اسم المنتج الكامل.

In [ ]:
DATE_WORDS = ("date", "time", "acquisition", "capture", "scene", "collect")

def print_tags(title, tags):
    print("=" * 80)
    print(title)
    print("=" * 80)
    if not tags:
        print("لا توجد Tags عامة داخل الملف.")
        return
    for key, value in sorted(tags.items()):
        marker = "  <-- احتمال تاريخ/وقت" if any(w in key.lower() for w in DATE_WORDS) else ""
        print(f"{key}: {value}{marker}")

print_tags("MS TAGS", ms_tags)
print()
print_tags("PAN TAGS", pan_tags)


## 5) مقارنة نظام الإحداثيات والتغطية الجغرافية

هذه الخلية تجيب مبدئيًا عن سؤالين:

1. هل الصورتان تستخدمان نظام الإحداثيات نفسه؟
2. هل تغطيان المنطقة نفسها أو يوجد بينهما تداخل كبير؟


In [ ]:
from rasterio.coords import BoundingBox

def area(bounds):
    return max(0, bounds.right - bounds.left) * max(0, bounds.top - bounds.bottom)

def intersection(a, b):
    left = max(a.left, b.left)
    bottom = max(a.bottom, b.bottom)
    right = min(a.right, b.right)
    top = min(a.top, b.top)
    if right <= left or top <= bottom:
        return None
    return BoundingBox(left, bottom, right, top)

with rasterio.open(MS_PATH) as ms, rasterio.open(PAN_PATH) as pan:
    same_crs = ms.crs == pan.crs
    print("نفس CRS:", same_crs)
    print("MS CRS :", ms.crs)
    print("PAN CRS:", pan.crs)

    if ms.crs is None or pan.crs is None:
        print("\nتحذير: واحدة من الصور لا تحتوي على CRS؛ لا يمكن تأكيد التطابق الجغرافي تلقائيًا.")
    else:
        # تحويل حدود PAN إلى CRS الخاص بـ MS عند الحاجة
        if same_crs:
            pan_bounds_in_ms = pan.bounds
        else:
            tb = transform_bounds(pan.crs, ms.crs, *pan.bounds, densify_pts=21)
            pan_bounds_in_ms = BoundingBox(*tb)

        inter = intersection(ms.bounds, pan_bounds_in_ms)

        print("\nMS bounds:", ms.bounds)
        print("PAN bounds in MS CRS:", pan_bounds_in_ms)

        if inter is None:
            print("\nالنتيجة: لا يوجد تداخل جغرافي حسب الـmetadata.")
        else:
            overlap_ms = area(inter) / area(ms.bounds) * 100
            overlap_pan = area(inter) / area(pan_bounds_in_ms) * 100
            print("\nIntersection:", inter)
            print(f"نسبة تغطية التداخل من مساحة MS : {overlap_ms:.2f}%")
            print(f"نسبة تغطية التداخل من مساحة PAN: {overlap_pan:.2f}%")

            if overlap_ms > 95 and overlap_pan > 95:
                print("النتيجة المبدئية: الصورتان تغطيان تقريبًا المنطقة نفسها.")
            elif overlap_ms > 70 and overlap_pan > 70:
                print("النتيجة المبدئية: يوجد تداخل كبير، لكن الحدود ليست متطابقة بالكامل.")
            else:
                print("النتيجة المبدئية: التداخل محدود، ويجب تحديد الجزء المشترك قبل التقطيع.")


## 6) حساب فرق الدقة المكانية

لو كانت قيمة `Scale Factor` قريبة من 4، فهذا يعني غالبًا أن بكسل MS واحد يقابله تقريبًا `4 × 4` بكسلات من PAN.

لكن وجود `MS` و`PAN` يعني أن المشكلة قد تكون **Pan-sharpening** وليست Super-Resolution تقليدية. سنحسم ذلك بعد رؤية عدد الـBands.

In [ ]:
with rasterio.open(MS_PATH) as ms, rasterio.open(PAN_PATH) as pan:
    if ms.crs is None or pan.crs is None:
        print("لا يمكن الاعتماد على حجم البكسل بدون CRS صحيح.")
    else:
        # المقارنة المباشرة تكون صحيحة فقط إذا كان الـCRS نفسه ووحداته نفسها
        if ms.crs == pan.crs:
            scale_x = abs(ms.transform.a) / abs(pan.transform.a)
            scale_y = abs(ms.transform.e) / abs(pan.transform.e)

            print(f"MS pixel size : {abs(ms.transform.a):.6f} × {abs(ms.transform.e):.6f}")
            print(f"PAN pixel size: {abs(pan.transform.a):.6f} × {abs(pan.transform.e):.6f}")
            print(f"Scale factor X: {scale_x:.4f}")
            print(f"Scale factor Y: {scale_y:.4f}")

            rounded_x = round(scale_x)
            rounded_y = round(scale_y)
            if abs(scale_x - rounded_x) < 0.05 and abs(scale_y - rounded_y) < 0.05:
                print(f"النسبة قريبة من ×{rounded_x}.")
            else:
                print("النسبة ليست عددًا صحيحًا واضحًا؛ قد نحتاج Resampling أو مراجعة الـmetadata.")
        else:
            print("الـCRS مختلف؛ سنحتاج توحيد الإسقاط قبل حساب Scale Factor بدقة.")


## 7) عرض المنطقة المشتركة بصريًا

الخلية التالية تعرض الجزء المشترك من الصورتين بحجم معاينة موحّد.

- صورة MS ستظهر RGB تقريبية إذا كان بها 3 Bands أو أكثر.
- صورة PAN غالبًا ستظهر Grayscale.
- الهدف الآن ليس الحكم على الجودة، بل التأكد أن الطرق والمباني والمعالم تقع في الأماكن نفسها.


In [ ]:
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds
from rasterio.enums import Resampling

def percentile_stretch(arr, p_low=2, p_high=98):
    arr = arr.astype(np.float32)
    out = np.zeros_like(arr, dtype=np.float32)
    if arr.ndim == 2:
        lo, hi = np.nanpercentile(arr, [p_low, p_high])
        if hi > lo:
            out = np.clip((arr - lo) / (hi - lo), 0, 1)
        return out

    for i in range(arr.shape[0]):
        band = arr[i]
        lo, hi = np.nanpercentile(band, [p_low, p_high])
        if hi > lo:
            out[i] = np.clip((band - lo) / (hi - lo), 0, 1)
    return out

with rasterio.open(MS_PATH) as ms, rasterio.open(PAN_PATH) as pan:
    if ms.crs is None or pan.crs is None:
        raise ValueError("لا يمكن استخراج الجزء المشترك تلقائيًا لأن CRS غير موجود.")

    if ms.crs == pan.crs:
        pan_bounds_in_ms = pan.bounds
    else:
        tb = transform_bounds(pan.crs, ms.crs, *pan.bounds, densify_pts=21)
        pan_bounds_in_ms = BoundingBox(*tb)

    inter = intersection(ms.bounds, pan_bounds_in_ms)
    if inter is None:
        raise ValueError("لا يوجد تداخل جغرافي بين الصورتين حسب الـmetadata.")

    # قراءة MS من الجزء المشترك
    ms_window = from_bounds(*inter, transform=ms.transform)
    if ms.count >= 3:
        # ترتيب افتراضي للعرض فقط. قد نعدله لاحقًا وفق ترتيب الـBands الحقيقي.
        ms_indexes = [1, 2, 3]
        ms_preview = ms.read(
            ms_indexes,
            window=ms_window,
            out_shape=(3, 600, 600),
            resampling=Resampling.bilinear
        )
        ms_preview = np.moveaxis(percentile_stretch(ms_preview), 0, -1)
    else:
        ms_preview = ms.read(
            1,
            window=ms_window,
            out_shape=(600, 600),
            resampling=Resampling.bilinear
        )
        ms_preview = percentile_stretch(ms_preview)

    # حدود التقاطع في CRS الخاص بـ PAN
    if ms.crs == pan.crs:
        inter_pan = inter
    else:
        tb2 = transform_bounds(ms.crs, pan.crs, *inter, densify_pts=21)
        inter_pan = BoundingBox(*tb2)

    pan_window = from_bounds(*inter_pan, transform=pan.transform)
    pan_preview = pan.read(
        1,
        window=pan_window,
        out_shape=(600, 600),
        resampling=Resampling.bilinear
    )
    pan_preview = percentile_stretch(pan_preview)

plt.figure(figsize=(8, 8))
plt.imshow(ms_preview, cmap="gray" if ms_preview.ndim == 2 else None)
plt.title("MS — common area")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 8))
plt.imshow(pan_preview, cmap="gray")
plt.title("PAN — common area")
plt.axis("off")
plt.show()


## 8) فحص مبدئي للانزياح بين الصورتين

هذا الفحص اختياري. يحاول مقارنة الحواف العامة بعد وضع الصورتين على شبكة واحدة.

> لأن MS وPAN مختلفتان طيفيًا، الرقم الناتج مؤشر مبدئي فقط، وليس حكمًا نهائيًا.

In [ ]:
from rasterio.warp import reproject
from skimage.filters import sobel
from skimage.registration import phase_cross_correlation

with rasterio.open(MS_PATH) as ms, rasterio.open(PAN_PATH) as pan:
    # قراءة Band واحدة من MS بحجم صغير
    target_h = min(ms.height, 1500)
    target_w = min(ms.width, 1500)

    ms_band = ms.read(
        1,
        out_shape=(target_h, target_w),
        resampling=Resampling.bilinear
    ).astype(np.float32)

    # Transform جديد بعد التصغير
    ms_transform_small = ms.transform * ms.transform.scale(
        ms.width / target_w,
        ms.height / target_h
    )

    pan_on_ms = np.zeros((target_h, target_w), dtype=np.float32)

    reproject(
        source=rasterio.band(pan, 1),
        destination=pan_on_ms,
        src_transform=pan.transform,
        src_crs=pan.crs,
        dst_transform=ms_transform_small,
        dst_crs=ms.crs,
        resampling=Resampling.bilinear
    )

    def standardize(x):
        valid = np.isfinite(x)
        if not valid.any():
            return np.zeros_like(x)
        med = np.nanmedian(x[valid])
        std = np.nanstd(x[valid])
        if std == 0:
            std = 1
        return np.nan_to_num((x - med) / std)

    ms_edges = sobel(standardize(ms_band))
    pan_edges = sobel(standardize(pan_on_ms))

    shift, error, phasediff = phase_cross_correlation(
        ms_edges,
        pan_edges,
        upsample_factor=10
    )

    print(f"Estimated shift [row, col] on the reduced MS grid: {shift}")
    print(f"Registration error indicator: {error:.6f}")
    print("\nالتفسير المبدئي:")
    print("- Shift قريب من [0, 0] جيد.")
    print("- Shift أكبر من 1–2 بكسل على شبكة MS يحتاج مراجعة بصرية ومحاذاة.")
    print("- اختلاف الطيف بين MS وPAN قد يجعل هذا القياس غير دقيق.")


## 9) إنشاء تقرير نصي مختصر

بعد تشغيل الخلايا، هذه الخلية تحفظ جدول المعلومات في ملف CSV داخل نفس مجلد الـNotebook.

In [ ]:
REPORT_PATH = Path("satellite_pair_metadata_report.csv")
info_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
print("Saved:", REPORT_PATH.resolve())


# ماذا ترسل بعد تشغيل الـNotebook؟

أرسل Screenshots أو انسخ ناتج الخلايا التالية:

1. جدول **الأبعاد وعدد الـBands والـCRS وحجم البكسل**.
2. نتيجة **نسبة التداخل**.
3. نتيجة **Scale Factor**.
4. صور المعاينة للـMS والـPAN.
5. الـMetadata التي تحتوي على تاريخ، إن ظهرت.
6. نتيجة الـEstimated Shift.

بعدها نقرر بدقة:

- هل الداتا **Super-Resolution pairs** أم **Pan-sharpening data**؟
- هل نحتاج Alignment؟
- ما حجم الـPatches؟
- هل نستخدم HAT كما هو أم نعدّل مسار المشروع؟
